In [1]:
import os
import sys
import sys

sys.path.append(os.path.abspath('..'))

WORKSPACE_DIR = "../lightrag_graphs"
os.makedirs(WORKSPACE_DIR, exist_ok=True)
ARCHIVO_ACTUAL = "../data/raw/book/christmas_carol_first_chapter_small.txt"

In [2]:
import os
import numpy as np
from lightrag import LightRAG
from lightrag.utils import wrap_embedding_func_with_attrs
from lightrag.llm.ollama import ollama_embed
from lightrag.llm.openai import openai_complete_if_cache


async def gemini_llm_wrapper(prompt, system_prompt=None, history_messages=[], **kwargs):
    kwargs["base_url"] = "https://generativelanguage.googleapis.com/v1beta/openai/"
    kwargs["api_key"] = os.getenv("GEMINI_API_KEY") 
    
    return await openai_complete_if_cache(
        model="gemini-2.5-flash",
        prompt=prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        **kwargs
    )

@wrap_embedding_func_with_attrs(
    embedding_dim=768,
    max_token_size=8192,
    model_name="nomic-embed-text"
)
async def embedding_func_local(texts: list[str]) -> np.ndarray:
    return await ollama_embed.func(texts, embed_model="nomic-embed-text")

rag = LightRAG(
    working_dir=WORKSPACE_DIR,
    llm_model_func=gemini_llm_wrapper,
    llm_model_max_async=1,
    embedding_func=embedding_func_local
)

INFO: [] Loaded graph from ../lightrag_graphs/graph_chunk_entity_relation.graphml with 109 nodes, 95 edges
INFO:nano-vectordb:Load (109, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': '../lightrag_graphs/vdb_entities.json'} 109 data
INFO:nano-vectordb:Load (95, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': '../lightrag_graphs/vdb_relationships.json'} 95 data
INFO:nano-vectordb:Load (4, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': '../lightrag_graphs/vdb_chunks.json'} 4 data


In [5]:
import sys
!{sys.executable} -m pip install nest-asyncio

zsh:1: no such file or directory: /Volumes/Toni


In [6]:
import nest_asyncio
nest_asyncio.apply()

with open(ARCHIVO_ACTUAL, "r", encoding="utf-8") as f:
    texto = f.read()
    
await rag.initialize_storages()
rag.insert(texto)

INFO: [] Process 40441 KV load full_docs with 0 records
INFO: [] Process 40441 KV load text_chunks with 0 records
INFO: [] Process 40441 KV load full_entities with 0 records
INFO: [] Process 40441 KV load full_relations with 0 records
INFO: [] Process 40441 KV load entity_chunks with 0 records
INFO: [] Process 40441 KV load relation_chunks with 0 records
INFO: [] Process 40441 KV load llm_response_cache with 0 records
INFO: [] Process 40441 doc status load doc_status with 0 records
INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-f4b91ba16666ba058dc5803bd7422903
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO: LLM func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: default:extract:ad5a0be74281dde6263eee2a56ab180a
INFO:  == LLM cache == saving: default:extract:7bf060053f53722954034984f98e7d07
INFO: Chunk 1 of 4

'insert_20260306_162410_c5cee4c0'

In [ ]:
import os
import networkx as nx
from pyvis.network import Network
import webbrowser

ruta_grafo = os.path.join("../lightrag_graphs", "graph_chunk_entity_relation.graphml")

if not os.path.exists(ruta_grafo):
    print("No encuentro el archivo del grafo. Revisa que la ruta sea correcta.")
else:    
    G = nx.read_graphml(ruta_grafo)
    
    net = Network(notebook=False, width="100%", height="100vh", bgcolor="#222222", font_color="white")
    net.from_nx(G)
    net.repulsion(node_distance=150, central_gravity=0.2, spring_length=200)
    
    archivo_html = "grafo_scrooge_fullscreen.html"
    net.save_graph(archivo_html)
    
    ruta_absoluta = "file://" + os.path.abspath(archivo_html)
    webbrowser.open(ruta_absoluta)

🎨 Generando el archivo interactivo...
✅ Grafo cargado con 109 Nodos y 95 Aristas.
🚀 ¡Revisa tu navegador web (Chrome/Safari)! Se ha abierto una pestaña nueva con el grafo a pantalla completa.


In [ ]:
pregunta = "¿Qué piensa Scrooge sobre la Navidad y qué frase repite?"

respuesta_local = rag.query(pregunta, param={"mode": "local"})
print(respuesta_local)

print("\n" + "="*50 + "\n")

respuesta_global = rag.query("¿Cuál es el ambiente general y el tono de la historia?", param={"mode": "global"})
print(respuesta_global)

INFO: [] Process 41209 KV load full_docs with 1 records
INFO: [] Process 41209 KV load text_chunks with 4 records
INFO: [] Process 41209 KV load full_entities with 1 records
INFO: [] Process 41209 KV load full_relations with 1 records
INFO: [] Process 41209 KV load entity_chunks with 109 records
INFO: [] Process 41209 KV load relation_chunks with 95 records
INFO: [] Process 41209 KV load llm_response_cache with 8 records
INFO: [] Process 41209 doc status load doc_status with 1 records


RuntimeError: This event loop is already running